In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, QED
from rdkit.Chem.rdMolDescriptors import CalcTPSA
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['text.color'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'

df = pd.read_csv("./data/output_from_KNIME/hit_selective_comparison.csv")
print(df.shape)
df.head()

In [ ]:
df2 = pd.read_csv("./data/output_from_KNIME/chem-space.csv")
df2.head()

In [ ]:
# --- 計算関数 ---
def calc_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        return [np.nan]*8
    return [
        Descriptors.MolWt(mol),
        Crippen.MolLogP(mol),
        Lipinski.NumHDonors(mol),
        Lipinski.NumHAcceptors(mol),
        CalcTPSA(mol),
        Lipinski.NumRotatableBonds(mol),
        Descriptors.FractionCSP3(mol),
        QED.qed(mol)
    ]

descriptor_names = ['MolecularWeight','LogP','HBD','HBA','TPSA','RotatableBonds','Fsp3','QED']
cont_vars = ['MolecularWeight','LogP','TPSA','Fsp3']
disc_vars = ['HBD','HBA','RotatableBonds']

criteria = 2  # ここを変えることで閾値を調整
df['selectivity'] = df['Num-hit-RNA/1000'].apply(lambda x: 'unselective' if x >= criteria else 'selective')

groups = ['selective', 'unselective']
palette = {'selective': 'Aquamarine', 'unselective': 'LightSeaGreen'}
df[descriptor_names] = df['SMILES'].apply(calc_descriptors).tolist()
df2[descriptor_names] = df2['SMILES'].apply(calc_descriptors).tolist()


df.head()

In [ ]:
df2.head()

In [ ]:
df_selective = df[df['selectivity'] == 'selective']
df_unselective = df[df['selectivity'] == 'unselective']

df_selective.head()


In [ ]:

descriptors = ['MolecularWeight', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotatableBonds', 'Fsp3', 'QED']

df2_rbind2 = df2[df2['type'] == 'R-BIND2']

print(f"Selective compounds: {len(df_selective)}")
print(f"Unselective compounds: {len(df_unselective)}")
print(f"R-BIND2 compounds: {len(df2_rbind2)}")

In [ ]:
# 各グループの平均値を計算
descriptors = ['MolecularWeight', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotatableBonds', 'Fsp3', 'QED']

selective_means = df_selective[descriptors].mean()
unselective_means = df_unselective[descriptors].mean()
rbind2_means = df2_rbind2[descriptors].mean()

# 1つのデータフレームにまとめ
mean_values_df = pd.DataFrame({
    'Selective': selective_means,
    'Unselective': unselective_means,
    'R-BIND2': rbind2_means
})

print("\n=== 平均値データフレーム ===")
print(mean_values_df.round(3))

print(f"\nデータフレームの形状: {mean_values_df.shape}")
print(f"列名: {list(mean_values_df.columns)}")
print(f"行名（記述子）: {list(mean_values_df.index)}")

# 各記述子での最大・最小グループを確認
print("\n=== 各記述子での特徴 ===")
for desc in descriptors:
    values = mean_values_df.loc[desc]
    max_group = values.idxmax()
    min_group = values.idxmin()
    max_val = values[max_group]
    min_val = values[min_group]
    diff = max_val - min_val
    print(f"{desc:15s}: 最大={max_group:11s} ({max_val:6.3f}), 最小={min_group:11s} ({min_val:6.3f}), 差={diff:6.3f}")

mean_values_df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from math import pi

descriptors = ['MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotatableBonds', 'Fsp3', 'QED']
selective_means = df_selective[['MolecularWeight', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotatableBonds', 'Fsp3', 'QED']].mean()
unselective_means = df_unselective[['MolecularWeight', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotatableBonds', 'Fsp3', 'QED']].mean()
all_data = pd.concat([selective_means, unselective_means], axis=1)
all_data.columns = ['Selective', 'Unselective']
normalized_data = {}
for group in ['Selective', 'Unselective']:
    values = []
    for i, desc in enumerate(['MolecularWeight', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotatableBonds', 'Fsp3', 'QED']):
        val = all_data.loc[desc, group]
        max_in_data = max(selective_means[desc], unselective_means[desc])
        
        normalized_val = (val / max_in_data) * 0.9
        values.append(normalized_val)
    normalized_data[group] = values

N = len(descriptors)
angles = [n / N * 2 * pi for n in range(N)]
angles += angles[:1]

colors = {
    'Selective': '#40E0D0',    # Turquoise（ターコイズ）
    'Unselective': '#B22222'   # Fire Brick（濃い赤）
}

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))
fig.patch.set_facecolor('white')

for group in ['Unselective', 'Selective']:  # 順番を変更
    values = normalized_data[group] + normalized_data[group][:1]
    ax.plot(angles, values, 'o-', linewidth=2.5, label=group, 
            color=colors[group], markersize=6)
    ax.fill(angles, values, alpha=0.15, color=colors[group])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([])  # デフォルトのラベルを削除
ax.set_ylim(0, 1.0)
ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels([])  # y軸ラベルを非表示
ax.grid(True, alpha=0.3, color='gray', linewidth=0.5)

label_distance = 1.15  # レーダーチャートから適度な距離
for angle, desc in zip(angles[:-1], descriptors):
    x = angle
    y = label_distance
    
    if angle == 0:  # 右
        ha, va = 'left', 'center'
    elif 0 < angle < pi/2:  # 右上
        ha, va = 'left', 'bottom'
    elif angle == pi/2:  # 上
        ha, va = 'center', 'bottom'
    elif pi/2 < angle < pi:  # 左上
        ha, va = 'right', 'bottom'
    elif angle == pi:  # 左
        ha, va = 'right', 'center'
    elif pi < angle < 3*pi/2:  # 左下
        ha, va = 'right', 'top'
    elif angle == 3*pi/2:  # 下
        ha, va = 'center', 'top'
    else:  # 右下
        ha, va = 'left', 'top'
    
    ax.text(x, y, desc, fontsize=11, fontweight='bold', 
            ha=ha, va=va, color='black')

ax.legend(loc='upper right', bbox_to_anchor=(1.15, 1.0), 
          fontsize=11, frameon=False)

plt.tight_layout()
plt.show()

print("\n=== 実測平均値 ===")
print("記述子          Selective   Unselective")
print("=" * 40)
for i, desc in enumerate(['MolecularWeight', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotatableBonds', 'Fsp3', 'QED']):
    sel_val = selective_means[desc]
    unsel_val = unselective_means[desc]
    print(f"{descriptors[i]:12s}    {sel_val:8.3f}     {unsel_val:8.3f}")

